---

# Unit 3 — Notebook 2: Re-Ranking & Query Expansion

## Introduction

Even the best retriever is imperfect. Two additional techniques are used in production Advanced RAG pipelines to push quality further:

1. **Query Expansion** — *Before retrieval*: Rewrite or expand the user's query so we retrieve better candidates.
2. **Re-Ranking** — *After retrieval*: Re-score the top-K retrieved documents with a more powerful (but slower) model, so the LLM sees only the truly relevant ones.

```mermaid
graph LR
    U["User Query"] --> QE["Query Expansion"]
    QE --> HR["Hybrid Retrieval (BM25+SBERT)"]
    HR --> RR["Re-Ranker (Cross-Encoder)"]
    RR --> LLM["LLM Generation"]
    LLM --> A["Answer"]
```

---

## Setup

In [1]:
%pip install python-dotenv rank-bm25 sentence-transformers langchain langchain-google-genai langchain-community langchain-huggingface numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Enter your Google API Key: ··········
Setup complete.


In [3]:
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# Shared corpus from Notebook 1 — extended for richer demo
corpus = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "BERT is a bidirectional encoder trained using masked language modelling.",
    "The BM25 algorithm ranks documents based on term frequency and inverse document frequency.",
    "Gradient descent is an optimization technique used to minimize the loss function.",
    "Neural networks learn by adjusting weights through backpropagation.",
    "Retrieval Augmented Generation combines a retriever with a language model to produce grounded answers.",
    "Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.",
    "Quantization reduces model size by representing weights in lower bit formats such as INT8.",
    "The attention mechanism computes a weighted sum of value vectors based on query-key similarity.",
    "Large language models are trained on massive text corpora to learn general-purpose representations.",
]

print(f"Corpus: {len(corpus)} documents loaded.")

Corpus: 10 documents loaded.


---

## Part 2a: Re-Ranking with Cross-Encoders

### The Problem with Bi-Encoders

In Notebook 1, SBERT is a **bi-encoder**: query and document are encoded *independently* and compared via cosine similarity.

The problem: the query and document never "see" each other during encoding — they are processed in isolation.

### The Cross-Encoder Solution

A **cross-encoder** takes the query and document *concatenated together* as a single input and outputs a single relevance score.

```mermaid
graph TD
    subgraph "Bi-Encoder (SBERT) — Fast"
        Q1["Query"] --> E1["Encoder"] --> V1["Query Vector"]
        D1["Document"] --> E2["Encoder"] --> V2["Doc Vector"]
        V1 & V2 --> CS["Cosine Similarity"]
    end
    subgraph "Cross-Encoder — Accurate"
        QD["[Query] [SEP] [Document]"] --> CE["BERT Encoder"] --> Score["Relevance Score (0-1)"]
    end
```

| Property | Bi-Encoder (SBERT) | Cross-Encoder |
|---|---|---|
| **Speed** | Very fast (pre-compute docs) | Slow (must run per query-doc pair) |
| **Quality** | Good | Significantly better |
| **Use case** | First-stage retrieval (top-100) | Second-stage re-ranking (top-10 → top-3) |

### The 2-Stage Pipeline Strategy

1. **Stage 1 — Fast Recall**: Use Hybrid Retrieval to get top-20 candidates.
2. **Stage 2 — Precise Re-ranking**: Use Cross-Encoder to re-score those 20 and pick top-3.

This gives you the speed of SBERT at scale + the accuracy of a cross-encoder for the final answer.

In [4]:
# Load cross-encoder model (much smaller than full BERT, fast enough for top-K re-ranking)
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Load bi-encoder for first-stage retrieval
bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("Models loaded.")
print("  Bi-encoder:    all-MiniLM-L6-v2 (first-stage retrieval)")
print("  Cross-encoder: ms-marco-MiniLM-L-6-v2 (re-ranking)")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded.
  Bi-encoder:    all-MiniLM-L6-v2 (first-stage retrieval)
  Cross-encoder: ms-marco-MiniLM-L-6-v2 (re-ranking)


In [5]:
# ---- Stage 1: Bi-Encoder Retrieval ----

query = "How does attention work in language models?"

doc_vecs   = bi_encoder.encode(corpus, convert_to_numpy=True)
doc_vecs   = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
query_vec  = bi_encoder.encode([query], convert_to_numpy=True)[0]
query_vec  = query_vec / np.linalg.norm(query_vec)

bi_scores  = doc_vecs @ query_vec
bi_ranked  = np.argsort(bi_scores)[::-1][:5]  # Top-5 from bi-encoder

print("STAGE 1 — Bi-Encoder Top-5 Candidates")
print("-" * 70)
for rank, idx in enumerate(bi_ranked, 1):
    print(f"  #{rank}  [score={bi_scores[idx]:.4f}] {corpus[idx]}")

STAGE 1 — Bi-Encoder Top-5 Candidates
----------------------------------------------------------------------
  #1  [score=0.5273] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2  [score=0.4970] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3  [score=0.3708] BERT is a bidirectional encoder trained using masked language modelling.
  #4  [score=0.3478] Transformers use self-attention mechanisms to process sequences in parallel.
  #5  [score=0.3418] Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.


In [6]:
# ---- Stage 2: Cross-Encoder Re-Ranking ----

# Build (query, doc) pairs for the cross-encoder
candidate_docs = [corpus[idx] for idx in bi_ranked]
query_doc_pairs = [[query, doc] for doc in candidate_docs]

# Cross-encoder scores (logit; higher = more relevant)
ce_scores = cross_encoder.predict(query_doc_pairs)

# Re-rank candidates
reranked_order = np.argsort(ce_scores)[::-1]

print("STAGE 2 — Cross-Encoder Re-Ranking")
print("-" * 70)
for rank, idx_in_candidates in enumerate(reranked_order, 1):
    original_doc_id = bi_ranked[idx_in_candidates]
    print(f"  #{rank}  [CE score={ce_scores[idx_in_candidates]:.4f}] {candidate_docs[idx_in_candidates]}")

print("\n--- Before vs After Re-Ranking ---")
print(f"{'Bi-Encoder Rank 1:':<30} {candidate_docs[0]}")
print(f"{'Cross-Encoder Rank 1:':<30} {candidate_docs[reranked_order[0]]}")

STAGE 2 — Cross-Encoder Re-Ranking
----------------------------------------------------------------------
  #1  [CE score=3.3522] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2  [CE score=-4.4683] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3  [CE score=-5.4407] Transformers use self-attention mechanisms to process sequences in parallel.
  #4  [CE score=-6.9237] BERT is a bidirectional encoder trained using masked language modelling.
  #5  [CE score=-10.4939] Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.

--- Before vs After Re-Ranking ---
Bi-Encoder Rank 1:             The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
Cross-Encoder Rank 1:          The attention mechanism computes a weighted sum of value vectors based on query-key similarity.


### What Just Happened?

The cross-encoder re-reads each (query, document) pair as a single combined input. This means:
- "attention" in the query can cross-attend to "attention mechanism" in the document at the token level.
- The model outputs a single relevance score (a logit — can be negative or large positive).

**Rule of thumb**: Use bi-encoders when your corpus has >10,000 documents (too slow to cross-encode all). Use the cross-encoder only on the top-20/50 returned by the bi-encoder.

---

## Part 2b: Query Expansion

### The Problem

Users write short, ambiguous queries. For example:
- `"python error"` — is this a snake biting an error? Or a code bug?
- `"model collapse"` — ML jargon that retrieval systems may not handle well.

**Query Expansion** rewrites or augments the query before retrieval.

### Technique 1: HyDE (Hypothetical Document Embedding)

**Idea**: Instead of embedding the query (which is short and vague), ask an LLM to *generate a hypothetical ideal answer*, then embed *that* to use as the search vector.

```mermaid
graph LR
    Q["Short Query"] --> LLM["LLM: Generate Hypothetical Answer"]
    LLM --> HA["Hypothetical Answer (longer, detailed)"]
    HA --> Embed["Embedding Model"]
    Embed --> VS["Vector Search"]
    VS --> Docs["Retrieved Documents"]
```

**Why does this work?** The hypothetical answer is written in the same language and style as the real documents, so the embedding is much closer to the ground-truth documents in vector space.

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.0)

# HyDE prompt: generate a hypothetical document that would answer this query
hyde_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a technical writer. Generate a single factual paragraph (3-5 sentences) that would directly answer the following question. Write it as if it were an excerpt from a textbook."),
    ("human", "{query}")
])

hyde_chain = hyde_prompt | llm | StrOutputParser()

query = "How does attention work in language models?"

# Without HyDE: embed the raw query
raw_query_vec = bi_encoder.encode([query], convert_to_numpy=True)[0]

# With HyDE: generate hypothetical doc, then embed it
hypothetical_doc = hyde_chain.invoke({"query": query})

print("Original Query:")
print(f"  '{query}'")
print(f"\nHypothetical Document (generated by Gemini):")
print(f"  {hypothetical_doc}")

Original Query:
  'How does attention work in language models?'

Hypothetical Document (generated by Gemini):
  Attention mechanisms in language models allow the model to dynamically weigh the importance of different parts of an input sequence when processing each element. This process involves computing a "query" vector for the current token, which is then compared against "key" vectors for all other tokens in the sequence to determine their relevance. The resulting similarity scores are normalized, typically via a softmax function, and then used as weights to create a weighted sum of "value" vectors associated with each token. This weighted sum forms a new, context-aware representation for the current token, effectively enabling the model to focus on the most pertinent information from the entire input to inform its understanding and generation. This mechanism is crucial for capturing long-range dependencies and intricate contextual relationships within text.


In [8]:
# Compare retrieval: raw query vs HyDE
hyde_vec = bi_encoder.encode([hypothetical_doc], convert_to_numpy=True)[0]
hyde_vec = hyde_vec / np.linalg.norm(hyde_vec)

raw_vec  = raw_query_vec / np.linalg.norm(raw_query_vec)

raw_scores  = doc_vecs @ raw_vec
hyde_scores = doc_vecs @ hyde_vec

raw_ranked  = np.argsort(raw_scores)[::-1][:3]
hyde_ranked = np.argsort(hyde_scores)[::-1][:3]

print(f"Query: '{query}'\n")
print("Raw Query Retrieval (Top 3):")
for r, idx in enumerate(raw_ranked, 1):
    print(f"  #{r} [score={raw_scores[idx]:.4f}] {corpus[idx]}")

print("\nHyDE Retrieval (Top 3):")
for r, idx in enumerate(hyde_ranked, 1):
    print(f"  #{r} [score={hyde_scores[idx]:.4f}] {corpus[idx]}")

Query: 'How does attention work in language models?'

Raw Query Retrieval (Top 3):
  #1 [score=0.5273] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2 [score=0.4970] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3 [score=0.3708] BERT is a bidirectional encoder trained using masked language modelling.

HyDE Retrieval (Top 3):
  #1 [score=0.6712] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  #2 [score=0.5350] Large language models are trained on massive text corpora to learn general-purpose representations.
  #3 [score=0.4125] Retrieval Augmented Generation combines a retriever with a language model to produce grounded answers.


### Technique 2: Multi-Query Retrieval

**Idea**: Generate multiple paraphrases/variations of the query, retrieve for each one, then take the union.

This handles the case where a single query phrasing misses relevant documents, but a different phrasing of the same question would find them.

```mermaid
graph LR
    Q["Original Query"] --> LLM["LLM: Generate 3 Query Variants"]
    LLM --> Q1["Variant 1"]
    LLM --> Q2["Variant 2"]
    LLM --> Q3["Variant 3"]
    Q1 --> R1["Retrieve top-3"]
    Q2 --> R2["Retrieve top-3"]
    Q3 --> R3["Retrieve top-3"]
    R1 & R2 & R3 --> Union["Deduplicated Union"]
    Union --> Rank["Re-Rank"]
```

In [9]:
# Multi-Query Retrieval — implemented manually (LangChain 1.x removed the old import path)
# The logic is identical to what MultiQueryRetriever did internally.

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Step 1: Use Gemini to generate 3 alternative phrasings of the query
multi_query_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are an AI assistant helping improve document retrieval.
Given a user question, generate 3 different paraphrases of that question.
Each paraphrase should approach the topic from a slightly different angle.
Return ONLY the 3 questions, one per line. No numbering, no extra text."""),
    ("human", "{query}")
])

multi_query_chain = multi_query_prompt | llm | StrOutputParser()

query = "How does attention work in language models?"

# Generate query variants
variants_raw = multi_query_chain.invoke({"query": query})
query_variants = [q.strip() for q in variants_raw.strip().split("\n") if q.strip()]
all_queries = [query] + query_variants[:3]  # original + up to 3 variants

print("Generated query variants:")
for i, q in enumerate(all_queries):
    label = "(original)" if i == 0 else f"(variant {i})"
    print(f"  {label}: {q}")

# Step 2: Retrieve top-3 for each query variant, then deduplicate
seen = set()
all_retrieved = []

for q in all_queries:
    q_vec   = bi_encoder.encode([q], convert_to_numpy=True)[0]
    q_vec   = q_vec / np.linalg.norm(q_vec)
    scores  = doc_vecs @ q_vec
    top_idx = np.argsort(scores)[::-1][:3]
    for idx in top_idx:
        text = corpus[idx]
        if text not in seen:
            seen.add(text)
            all_retrieved.append({"doc_id": int(idx), "text": text, "from_query": q})

print(f"\nMulti-Query retrieved {len(all_retrieved)} unique documents (union of all variants):")
for doc in all_retrieved:
    print(f"  [doc_{doc['doc_id']}] {doc['text']}")

Generated query variants:
  (original): How does attention work in language models?
  (variant 1): Describe the mechanism of attention within language models.
  (variant 2): What role do attention mechanisms play in the operation of language models?
  (variant 3): How do language models leverage attention to process information?

Multi-Query retrieved 5 unique documents (union of all variants):
  [doc_8] The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
  [doc_9] Large language models are trained on massive text corpora to learn general-purpose representations.
  [doc_1] BERT is a bidirectional encoder trained using masked language modelling.
  [doc_0] Transformers use self-attention mechanisms to process sequences in parallel.
  [doc_6] Fine-tuning adapts a pre-trained model to a specific downstream task using task-specific data.


### HyDE vs Multi-Query: When to Use Which?

| Technique | Best For | Weakness |
|---|---|---|
| **HyDE** | Short, vague queries; when query language ≠ document language | LLM hallucination in the hypothetical doc can mislead retrieval |
| **Multi-Query** | Covering multiple angles of a question; exploratory search | Higher latency (multiple LLM calls); may retrieve noisy documents |

---

## Part 2c: Full Production Pipeline — Query Expansion → Hybrid → Re-Rank → Generate

Now we chain everything together in a single LCEL pipeline.

In [10]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

# --- Reuse HybridRetriever from Notebook 1 ---
class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25  = BM25Okapi(tokenized)
        sbert      = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
        doc_vecs   = sbert.encode(corpus, convert_to_numpy=True)
        self.doc_vecs = doc_vecs / np.linalg.norm(doc_vecs, axis=1, keepdims=True)
        self.sbert = sbert

    def retrieve(self, query, top_k=5):
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranked = np.argsort(bm25_scores)[::-1]
        bm25_ranks  = {int(d): r+1 for r, d in enumerate(bm25_ranked)}

        q_vec       = self.sbert.encode([query], convert_to_numpy=True)[0]
        q_vec       = q_vec / np.linalg.norm(q_vec)
        sbert_scores = self.doc_vecs @ q_vec
        sbert_ranked = np.argsort(sbert_scores)[::-1]
        sbert_ranks  = {int(d): r+1 for r, d in enumerate(sbert_ranked)}

        rrf = {d: 1/(self.k+bm25_ranks[d]) + 1/(self.k+sbert_ranks[d]) for d in range(len(self.corpus))}
        final = sorted(rrf, key=rrf.get, reverse=True)[:top_k]
        return [self.corpus[d] for d in final]

hybrid = HybridRetriever(corpus)

# --- Step functions ---

def expand_query(query: str) -> str:
    """Use HyDE to expand the query."""
    return hyde_chain.invoke({"query": query})

def hybrid_retrieve(expanded_query: str) -> list[str]:
    """Retrieve top-5 using Hybrid (BM25+SBERT+RRF)."""
    return hybrid.retrieve(expanded_query, top_k=5)

def rerank(data: dict) -> list[str]:
    """Re-rank retrieved docs using cross-encoder."""
    query = data["original_query"]
    docs  = data["candidates"]
    pairs  = [[query, doc] for doc in docs]
    scores = cross_encoder.predict(pairs)
    ranked = np.argsort(scores)[::-1][:3]  # Keep top-3
    return [docs[i] for i in ranked]

def format_context(docs: list[str]) -> str:
    return "\n\n".join(f"[Document {i+1}]\n{doc}" for i, doc in enumerate(docs))

print("Pipeline components defined.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Pipeline components defined.


In [11]:
# --- Final Answer Generation Prompt ---
generation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a knowledgeable AI assistant. Answer the user's question using ONLY the provided context documents.
If the answer is not in the context, say 'I don't have enough information to answer this.'
Be concise and precise.

Context:
{context}"""),
    ("human", "{question}")
])

# --- Full pipeline using LCEL ---
def run_advanced_rag(user_query: str):
    print(f"Query: '{user_query}'")
    print("=" * 70)

    # Step 1: Query Expansion (HyDE)
    expanded = expand_query(user_query)
    print(f"[1] HyDE Expansion: {expanded[:150]}...")

    # Step 2: Hybrid Retrieval on expanded query
    candidates = hybrid_retrieve(expanded)
    print(f"\n[2] Hybrid Retrieval ({len(candidates)} candidates):")
    for i, c in enumerate(candidates, 1):
        print(f"     #{i}: {c[:80]}")

    # Step 3: Cross-Encoder Re-Ranking
    top_docs = rerank({"original_query": user_query, "candidates": candidates})
    print(f"\n[3] After Re-Ranking (top 3):")
    for i, d in enumerate(top_docs, 1):
        print(f"     #{i}: {d}")

    # Step 4: LLM Generation
    context = format_context(top_docs)
    chain   = generation_prompt | llm | StrOutputParser()
    answer  = chain.invoke({"context": context, "question": user_query})

    print(f"\n[4] Final Answer:")
    print(f"     {answer}")
    return answer

# --- Run it ---
run_advanced_rag("How does attention work in language models?")

Query: 'How does attention work in language models?'
[1] HyDE Expansion: Attention mechanisms in language models enable the model to dynamically weigh the importance of different parts of an input sequence when processing a...

[2] Hybrid Retrieval (5 candidates):
     #1: The attention mechanism computes a weighted sum of value vectors based on query-
     #2: Large language models are trained on massive text corpora to learn general-purpo
     #3: Fine-tuning adapts a pre-trained model to a specific downstream task using task-
     #4: Retrieval Augmented Generation combines a retriever with a language model to pro
     #5: Transformers use self-attention mechanisms to process sequences in parallel.

[3] After Re-Ranking (top 3):
     #1: The attention mechanism computes a weighted sum of value vectors based on query-key similarity.
     #2: Large language models are trained on massive text corpora to learn general-purpose representations.
     #3: Transformers use self-attention mec

"I don't have enough information to answer this."

In [12]:
# Try a second query
run_advanced_rag("What is the purpose of backpropagation?")

Query: 'What is the purpose of backpropagation?'
[1] HyDE Expansion: Backpropagation is a fundamental algorithm used to efficiently train artificial neural networks. Its primary purpose is to calculate the gradient of t...

[2] Hybrid Retrieval (5 candidates):
     #1: Gradient descent is an optimization technique used to minimize the loss function
     #2: Neural networks learn by adjusting weights through backpropagation.
     #3: The attention mechanism computes a weighted sum of value vectors based on query-
     #4: The BM25 algorithm ranks documents based on term frequency and inverse document 
     #5: Fine-tuning adapts a pre-trained model to a specific downstream task using task-

[3] After Re-Ranking (top 3):
     #1: Neural networks learn by adjusting weights through backpropagation.
     #2: Gradient descent is an optimization technique used to minimize the loss function.
     #3: The attention mechanism computes a weighted sum of value vectors based on query-key similarity

'Backpropagation is used for adjusting weights in neural networks.'

---

## Summary

### The Full Advanced RAG Stack

```
User Query
    │
    ▼
Query Expansion (HyDE / Multi-Query)
    │  → Makes the query richer and closer to document language
    ▼
Hybrid Retrieval (BM25 + SBERT + RRF)
    │  → Combines keyword and semantic search
    ▼
Cross-Encoder Re-Ranking
    │  → Selects the truly relevant documents from candidates
    ▼
LLM Generation (Gemini)
    │  → Produces a grounded, accurate answer
    ▼
Answer
```

### Comparison of Approaches

| Stage | Naïve RAG | Advanced RAG |
|---|---|---|
| Query | Raw user input | HyDE expanded / multi-phrasing |
| Retrieval | Dense only (cosine) | Hybrid (BM25 + SBERT + RRF) |
| Candidate filtering | Top-K by embedding score | Cross-encoder re-ranking |
| Generation | All top-K in context | Only highest-quality docs |

**Next**: Notebook 3 will shift from the retrieval side to the **generation side** — fine-tuning, quantization, and Mixture of Experts.